# Google Play Store - Data Cleaning for Tableau
**Prepared by:** Shubham Shah  
**Tools used:** Python, NumPy, Pandas

In this notebook I clean the Google Play Store data step by step and save the clean csv file, which I will use to build the Tableau dashboard.

**Input:** `googleplaystore.csv` (keep it in the same folder as this notebook)  
**Output:** `playstore_clean_data.csv` saved in `D:\Playestore_clean_data`

## Step 1: Import Libraries and Load the Data

In [3]:
import os
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv('googleplaystore.csv')

## Step 2: Understanding the Data

In [6]:
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [7]:
df.shape

(10841, 13)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


In [9]:
df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

## Step 3: Data Cleaning

### 3.1 Removing exact duplicate rows

In [12]:
df.duplicated().sum()

483

In [13]:
df = df.drop_duplicates().copy()
df.shape

(10358, 13)

### 3.2 Removing the wrong row
One row has all its values shifted by one column (its Category shows as `1.9`), so it is not a real app record.

In [15]:
df[df['Category'] == '1.9']

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


In [16]:
df = df[df['Category'] != '1.9'].copy()
df.shape

(10357, 13)

### 3.3 Converting the data types
Reviews, Installs, Price and Size are stored as text, so I convert them into numbers. `Last Updated` is converted into a date.

In [18]:
# Reviews: text -> number
df['Reviews'] = pd.to_numeric(df['Reviews'])

In [19]:
# Installs: '10,000+' -> 10000
# I keep the original text in a new column (Install_Band), it is useful as a label in Tableau
df['Install_Band'] = df['Installs']

df['Installs'] = df['Installs'].str.replace(',', '', regex=False).str.replace('+', '', regex=False)
df['Installs'] = pd.to_numeric(df['Installs'])

In [20]:
# Price: '$4.99' -> 4.99
df['Price'] = df['Price'].str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'])

In [21]:
# Size: '19M' -> 19, '201k' -> 0.2 (in MB), 'Varies with device' -> blank
def convert_size(size):
    if size.endswith('M'):
        return float(size[:-1])
    elif size.endswith('k'):
        return float(size[:-1]) / 1024
    else:
        return np.nan

df['Size_MB'] = df['Size'].apply(convert_size).round(2)
df[['Size', 'Size_MB']].head(10)

,Size,Size_MB
0,19M,19.0
1,14M,14.0
2,8.7M,8.7
3,25M,25.0
4,2.8M,2.8
5,5.6M,5.6
6,19M,19.0
7,29M,29.0
8,33M,33.0
9,3.1M,3.1


In [22]:
# Last Updated: text -> date
df['Last Updated'] = pd.to_datetime(df['Last Updated'])
df['Last Updated'].head()

0   2018-01-07
1   2018-01-15
2   2018-08-01
3   2018-06-08
4   2018-06-20
Name: Last Updated, dtype: datetime64[ns]

### 3.4 Removing repeated apps
Some apps appear more than once with different review counts. I keep only one row for every app - the one with the most reviews.

In [24]:
df['App'].duplicated().sum()

698

In [25]:
df = df.sort_values('Reviews', ascending=False)
df = df.drop_duplicates(subset='App', keep='first')
df = df.sort_index().reset_index(drop=True)
df.shape

(9659, 15)

### 3.5 Handling missing values

In [27]:
df.isnull().sum()

App                  0
Category             0
Rating            1463
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       0
Genres               0
Last Updated         0
Current Ver          8
Android Ver          2
Install_Band         0
Size_MB           1228
dtype: int64

In [28]:
# Type: only one app is missing it and its price is 0, so it is a Free app
df['Type'] = df['Type'].fillna('Free')

# text columns
df['Content Rating'] = df['Content Rating'].fillna('Unknown')
df['Current Ver'] = df['Current Ver'].fillna('Unknown')
df['Android Ver'] = df['Android Ver'].fillna('Unknown')

I did **not** fill `Rating` and `Size_MB`. A blank is more honest than a made-up value, and Tableau ignores blanks when it calculates averages.

### 3.6 Standardizing the text

In [31]:
# Category: 'ART_AND_DESIGN' -> 'Art And Design'
df['Category'] = df['Category'].str.replace('_', ' ').str.title().str.strip()

# Genres: take the first genre ('Art & Design;Pretend Play' -> 'Art & Design')
df['Genres'] = df['Genres'].str.strip()
df['Main_Genre'] = df['Genres'].str.split(';').str[0].str.strip()

df['App'] = df['App'].str.strip()
df['Type'] = df['Type'].str.strip()
df['Content Rating'] = df['Content Rating'].str.strip()

In [32]:
df[['Category', 'Genres', 'Main_Genre']].head()

,Category,Genres,Main_Genre
0,Art And Design,Art & Design,Art & Design
1,Art And Design,Art & Design,Art & Design
2,Art And Design,Art & Design,Art & Design
3,Art And Design,Art & Design;Creativity,Art & Design
4,Art And Design,Art & Design,Art & Design


### 3.7 Extra columns for Tableau

In [34]:
df['Updated_Year'] = df['Last Updated'].dt.year
df['Updated_Month_No'] = df['Last Updated'].dt.month
df['Updated_Month'] = df['Last Updated'].dt.month_name()

## Step 4: Final Check

In [36]:
# neat column order (the old text Size column is dropped)
df = df[['App', 'Category', 'Rating', 'Reviews', 'Size_MB', 'Installs', 'Install_Band',
         'Type', 'Price', 'Content Rating', 'Genres', 'Main_Genre', 'Last Updated',
         'Updated_Year', 'Updated_Month_No', 'Updated_Month', 'Current Ver', 'Android Ver']]

df.shape

(9659, 18)

In [37]:
df.head()

,App,Category,Rating,Reviews,Size_MB,Installs,Install_Band,Type,Price,Content Rating,Genres,Main_Genre,Last Updated,Updated_Year,Updated_Month_No,Updated_Month,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,Art And Design,4.1,159,19.0,10000,"10,000+",Free,0.0,Everyone,Art & Design,Art & Design,2018-01-07,2018,1,January,1.0.0,4.0.3 and up
1,"U Launcher Lite – FREE Live Cool Themes, Hide ...",Art And Design,4.7,87510,8.7,5000000,"5,000,000+",Free,0.0,Everyone,Art & Design,Art & Design,2018-08-01,2018,8,August,1.2.4,4.0.3 and up
2,Sketch - Draw & Paint,Art And Design,4.5,215644,25.0,50000000,"50,000,000+",Free,0.0,Teen,Art & Design,Art & Design,2018-06-08,2018,6,June,Varies with device,4.2 and up
3,Pixel Draw - Number Art Coloring Book,Art And Design,4.3,967,2.8,100000,"100,000+",Free,0.0,Everyone,Art & Design;Creativity,Art & Design,2018-06-20,2018,6,June,1.1,4.4 and up
4,Paper flowers instructions,Art And Design,4.4,167,5.6,50000,"50,000+",Free,0.0,Everyone,Art & Design,Art & Design,2017-03-26,2017,3,March,1.0,2.3 and up


In [38]:
df.dtypes

App                         object
Category                    object
Rating                     float64
Reviews                      int64
Size_MB                    float64
Installs                     int64
Install_Band                object
Type                        object
Price                      float64
Content Rating              object
Genres                      object
Main_Genre                  object
Last Updated        datetime64[ns]
Updated_Year                 int32
Updated_Month_No             int32
Updated_Month               object
Current Ver                 object
Android Ver                 object
dtype: object

In [39]:
df.isnull().sum()

App                    0
Category               0
Rating              1463
Reviews                0
Size_MB             1228
Installs               0
Install_Band           0
Type                   0
Price                  0
Content Rating         0
Genres                 0
Main_Genre             0
Last Updated           0
Updated_Year           0
Updated_Month_No       0
Updated_Month          0
Current Ver            0
Android Ver            0
dtype: int64

Only `Rating` and `Size_MB` have blanks, and that is on purpose (explained in step 3.5).

## Step 5: Save the Clean Data

In [42]:
save_folder = r'D:\Playestore_clean_data'

# create the folder if it does not exist
os.makedirs(save_folder, exist_ok=True)

save_path = os.path.join(save_folder, 'playstore_clean_data.csv')
df.to_csv(save_path, index=False, encoding='utf-8-sig')

print('Clean data saved at:', save_path)

Clean data saved at: D:\Playestore_clean_data\playstore_clean_data.csv


The clean file is saved. In Tableau, connect to it using **Text file** and check that `Last Updated` is a date and `Rating`, `Reviews`, `Size_MB`, `Installs` and `Price` are numbers.

*- Shubham Shah*